# Round 2 Data - Klebsiella

In [1]:
import os, sys, re
import pandas as pd
from paths import *
from Bio import SeqIO

klebsiella_raw_dir = os.path.join(raw_data_path, 'phagehost_KU/data2_raw')
bact_raw_dir = os.path.join(klebsiella_raw_dir, 'Bakterier')
phage_raw_dir = os.path.join(klebsiella_raw_dir, 'Fager/output_phold')
#print(os.listdir(klebsiella_raw_dir))
print(os.listdir(bact_raw_dir))
print(os.listdir(phage_raw_dir))

### Functions ###
def concatenate_fastas_to_one(in_dir, output_file):
    """Concatenate multiple FASTA files into one FASTA file."""
    with open(output_file, 'w') as out_f:
        for filename in os.listdir(in_dir):
            if filename.endswith('.fasta'):
                fasta_file = os.path.join(in_dir, filename)
                for record in SeqIO.parse(fasta_file, "fasta"):
                    out_f.write(f'>{record.id}\n')
                for i in range(0, len(record.seq), 60):
                    out_f.write(str(record.seq[i:i+60]) + '\n')

##################

['KUdata21042026_1subset', '.DS_Store', 'KUdata20042026_4subsets', 'assemblies']
['phold_all_cds_functions.tsv', '.DS_Store', 'phold_prostT5_3di_all_probabilities.json', 'phold_run_1774895486.9737782.log', 'phold_3di.fasta', 'separate_gbks', 'phold_prostT5_3di_mean_probabilities.csv', 'phold.gbk', 'phold_aa.fasta', 'logs', 'sub_db_tophits', 'phold_per_cds_predictions.tsv']


## Extract Bact Genomes

In [44]:
bact_raw_assemblies = os.path.join(bact_raw_dir, 'assemblies')

def count_bp_in_fasta(fasta_file):
    circular_bp = 0
    linear_bp = 0
    with open(fasta_file, 'r') as f:
        for line in f:
            if line.startswith('>'):
                match = re.search(r"length=(\d+)", line)
                if match:
                    length_value = int(match.group(1))
                    if 'circular' in line:
                        circular_bp += length_value
                    else:
                        linear_bp += length_value
                else:
                    print(f"Regex failed to find length in line: {line}")
    return circular_bp, linear_bp

def concatenate_entries_in_fasta(fasta_file, output_file, include_in_header=None):
    sequences = []
    for record in SeqIO.parse(fasta_file, "fasta"):
        sequences.append(str(record.seq))
    
    concatenated_seq = ''.join(sequences)
    filename = os.path.basename(fasta_file).replace('.fasta', '')
    filename = filename.split('.')[0]  # Remove any additional extensions
    if include_in_header:
        filename += f'_{include_in_header}'
    with open(output_file, 'w') as out_f:
        out_f.write(f'>{filename}\n')
        for i in range(0, len(concatenated_seq), 60):
            out_f.write(concatenated_seq[i:i+60] + '\n')


In [45]:
for dir in os.listdir(bact_raw_assemblies):
    if dir == '.DS_Store':
        continue
    circular_bp = 0
    linear_bp = 0

    outdir = os.path.join(raw_data_path, 'phagehost_KU/data2_klebbacts/')
    os.makedirs(outdir, exist_ok=True)

    curr_dir = os.path.join(bact_raw_assemblies, dir)
    print(os.listdir(curr_dir))
    for file in os.listdir(curr_dir):
        if file.endswith('.fasta'):
            print(f"\nProcessing {file} in {curr_dir}")
            circular_bp, linear_bp = count_bp_in_fasta(os.path.join(curr_dir, file))
            concatenate_entries_in_fasta(os.path.join(curr_dir, file), os.path.join(outdir, f'{file}'), include_in_header=f"circular_bp={circular_bp}_linear_bp={linear_bp}")
            print(f"Circular bp: {circular_bp}")
            print(f"Linear bp: {linear_bp}")



['Kp_KU6.autocycler_medaka_polypolish.fasta', 'Kp_KU12.autocycler_medaka_polypolish.fasta', 'Kp_KU8.autocycler_medaka_polypolish.fasta', 'Kp_KU3.autocycler_medaka_polypolish.fasta', 'Kp_KU6.autocycler_medaka_polypolish_concatenated.fasta', 'Kp_KU4.autocycler_medaka_polypolish.fasta', 'Kp_KU9.autocycler_medaka_polypolish.fasta']

Processing Kp_KU6.autocycler_medaka_polypolish.fasta in /Users/asbjornhansen/PredictPhagePPI/raw_data/phagehost_KU/data2_raw/Bakterier/assemblies/1subset
Circular bp: 246199
Linear bp: 4868163

Processing Kp_KU12.autocycler_medaka_polypolish.fasta in /Users/asbjornhansen/PredictPhagePPI/raw_data/phagehost_KU/data2_raw/Bakterier/assemblies/1subset
Circular bp: 68599
Linear bp: 5202832

Processing Kp_KU8.autocycler_medaka_polypolish.fasta in /Users/asbjornhansen/PredictPhagePPI/raw_data/phagehost_KU/data2_raw/Bakterier/assemblies/1subset
Circular bp: 248714
Linear bp: 3980017

Processing Kp_KU3.autocycler_medaka_polypolish.fasta in /Users/asbjornhansen/PredictPha

#### Concatenate fastas to one

In [47]:
concatenate_fastas_to_one(outdir, os.path.join(raw_data_path, 'phagehost_KU/data2_bacts.fasta'))

## Extract Phage Genomes

In [2]:
phage_raw_gbks = os.path.join(phage_raw_dir, 'separate_gbks/')
files = sorted(os.listdir(phage_raw_gbks))
print(f"Files in {phage_raw_gbks}: {len(files)}")

### Functions ###

def extract_sequence_from_gbk(gbk_file):
    """Extract the nucleotide sequence from a GenBank file."""
    for record in SeqIO.parse(gbk_file, "genbank"):
        return str(record.seq)
    return None


#################

outdir = os.path.join(raw_data_path, 'phagehost_KU/data2_phages/')
os.makedirs(outdir, exist_ok=True)

Files in /Users/asbjornhansen/PredictPhagePPI/raw_data/phagehost_KU/data2_raw/Fager/output_phold/separate_gbks/: 54


In [5]:
for file in files:
    if file.endswith('.gbk'):
        #print(f"Processing {file} in {phage_raw_gbks}")
        gbk_path = os.path.join(phage_raw_gbks, file)
        sequence = extract_sequence_from_gbk(gbk_path)
        #print(sequence[:100])  # Print the first 100 bases to verify
        #print(f"Length of sequence: {len(sequence) if sequence else 'No sequence found'}")
        if "N" in sequence:
            print(f"Sequence contains Ns: {file}")
            print(f"Length before removing Ns: {len(sequence)}")
            sequence = sequence.replace('N', '')  # Remove Ns if present
            print(f"Length after removing Ns: {len(sequence)}\n")
        if sequence:
            filename = os.path.basename(file).replace('.gbk', '.fasta')
            with open(os.path.join(outdir, filename), 'w') as out_f:
                out_f.write(f'>{filename.replace(".fasta", "")}\n')
                for i in range(0, len(sequence), 60):
                    out_f.write(sequence[i:i+60] + '\n')


Sequence contains Ns: Balbinus_Host_11.gbk
Length before removing Ns: 34526
Length after removing Ns: 34524

Sequence contains Ns: Etui_Host_1.gbk
Length before removing Ns: 44258
Length after removing Ns: 44221

Sequence contains Ns: Leonitus_Host_4.gbk
Length before removing Ns: 18272
Length after removing Ns: 18270

Sequence contains Ns: Nepotimus_Host_11.gbk
Length before removing Ns: 176296
Length after removing Ns: 176294

Sequence contains Ns: Septimius_Host_10.gbk
Length before removing Ns: 44059
Length after removing Ns: 44015

Sequence contains Ns: Skandal_Host_13.gbk
Length before removing Ns: 8254
Length after removing Ns: 8125

Sequence contains Ns: Trebonianus_Host_11.gbk
Length before removing Ns: 243513
Length after removing Ns: 243472



#### Concatenate fastas to one

In [6]:
concatenate_fastas_to_one(outdir, os.path.join(raw_data_path, 'phagehost_KU/data2_phages.fasta'))

## Clean EOP (hostrange)
Clean and transpose EOP to match the hostrange from the previous data source 

In [ ]:
### Load and check data quality ###
# Data is in the correct format, but PFU values are from 0 to 1, which is not the same for the previous hostrange that went well above 1.
EOP_df = pd.read_excel(os.path.join(klebsiella_raw_dir, 'EOP.xlsx'))
display(EOP_df.head())

print(f"Dtypes in EOP data:\n{EOP_df.dtypes}")

# Concatenate "Phage name" and "no." columns to one column
EOP_df.insert(0, 'Phage', EOP_df['Phage name'] + '_' + EOP_df['no.'].astype(str).str.split('.').str[0] + "_host" + EOP_df["Original host"].astype(str))  # Keep only the part before the decimal point
EOP_df.index = EOP_df['Phage']  # Set the new "Phage" column as the index
EOP_df.drop(columns=['Phage', 'Phage name', 'no.', 'Original host'], inplace=True) #remove the original "Phage name" and "no." columns
EOP_df = EOP_df.transpose()  # Transpose the DataFrame to have phages as columns and hosts as rows

display(EOP_df)

,Phage name,no.,Original host,Host 1,Host 2,Host 3,Host 4,Host 5,Host 6,Host 7,Host 8,Host 9,Host 10,Host 11,Host 12,Host 13
0,Grebano,1.0,1,1.0,0.0,0.000000,0.733333,0.000217,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000
1,Ravello,2.0,1,1.0,0.0,0.000000,0.000000,0.000008,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000
2,Etui,3.0,1,1.0,0.0,0.000000,0.700000,0.027500,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000
3,Maxentius,4.0,2,0.0,1.0,0.076087,0.000000,0.000000,0.0,0.119565,0.295714,0.0,0.0,0.077174,0.0,0.445652
4,Licinius,5.0,2,0.0,1.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000


Dtypes in EOP data:
Phage name        object
no.              float64
Original host      int64
Host 1           float64
Host 2           float64
Host 3           float64
Host 4           float64
Host 5           float64
Host 6           float64
Host 7           float64
Host 8           float64
Host 9           float64
Host 10          float64
Host 11          float64
Host 12          float64
Host 13          float64
dtype: object


Phage,Grebano_1_host1,Ravello_2_host1,Etui_3_host1,Maxentius_4_host2,Licinius_5_host2,Jovian_6_host2,Arcadius_7_host2,Avitus_8_host3,Marcian_9_host3,Libius_10_host3,...,Trebonianus_41_host11,Skandal_42_host13,Balder_43_host13,Herennius_44_host13,Silbannacus_45_host13,Volusianus_46_host13,Galleinus_47_host13,Salolinus_48_host13,Carinus_49_host13,Galerius_50_host13
Host 1,1.000000,1.000000,1.0000,0.000000,0.0,0.00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
Host 2,0.000000,0.000000,0.0000,1.000000,1.0,1.00,1.000000,0.047619,0.000061,0.000042,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
Host 3,0.000000,0.000000,0.0000,0.076087,0.0,0.44,0.102222,1.000000,1.000000,1.000000,...,0.000000,0.967742,0.000636,0.635135,0.000671,0.000825,0.010000,0.000609,0.002632,0.0014
Host 4,0.733333,0.000000,0.7000,0.000000,0.0,0.00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
Host 5,0.000217,0.000008,0.0275,0.000000,0.0,0.00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
Host 6,0.000000,0.000000,0.0000,0.000000,0.0,0.00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.003226,0.909091,0.000743,0.002429,0.000000,0.373333,0.260870,0.438596,0.0000
Host 7,0.000000,0.000000,0.0000,0.119565,0.0,0.00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
Host 8,0.000000,0.000000,0.0000,0.295714,0.0,0.00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
Host 9,0.000000,0.000000,0.0000,0.000000,0.0,0.00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000032,0.008409,0.004459,0.001286,0.000000,0.000000,0.000000,0.002632,0.0000
Host 10,0.000000,0.000000,0.0000,0.000000,0.0,0.00,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000


Save to raw_dir

In [72]:
EOP_df.to_excel(os.path.join(raw_data_path, 'phagehost_KU/data2_EOP.xlsx'))

#### Check hostrange functions

In [2]:
from io_operations import call_hostrange_df
from manipulations import binarize_host_range, hostrange_df_to_dict

bact_lookup, EOP_hostrange_df = call_hostrange_df(os.path.join(raw_data_path, 'phagehost_KU/data2_EOP.xlsx'), sheet_name="Sheet1", data2=True)
display(EOP_hostrange_df)

,phage,Grebano_1_host1,Ravello_2_host1,Etui_3_host1,Maxentius_4_host2,Licinius_5_host2,Jovian_6_host2,Arcadius_7_host2,Avitus_8_host3,Marcian_9_host3,...,Trebonianus_41_host11,Skandal_42_host13,Balder_43_host13,Herennius_44_host13,Silbannacus_45_host13,Volusianus_46_host13,Galleinus_47_host13,Salolinus_48_host13,Carinus_49_host13,Galerius_50_host13
0,Host 1,1.000000,1.000000,1.0000,0.000000,0,0.00,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
1,Host 2,0.000000,0.000000,0.0000,1.000000,1,1.00,1.000000,0.047619,0.000061,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
2,Host 3,0.000000,0.000000,0.0000,0.076087,0,0.44,0.102222,1.000000,1.000000,...,0.000000,0.967742,0.000636,0.635135,0.000671,0.000825,0.010000,0.000609,0.002632,0.0014
3,Host 4,0.733333,0.000000,0.7000,0.000000,0,0.00,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
4,Host 5,0.000217,0.000008,0.0275,0.000000,0,0.00,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
5,Host 6,0.000000,0.000000,0.0000,0.000000,0,0.00,0.000000,0.000000,0.000000,...,0.000000,0.003226,0.909091,0.000743,0.002429,0.000000,0.373333,0.260870,0.438596,0.0000
6,Host 7,0.000000,0.000000,0.0000,0.119565,0,0.00,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
7,Host 8,0.000000,0.000000,0.0000,0.295714,0,0.00,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000
8,Host 9,0.000000,0.000000,0.0000,0.000000,0,0.00,0.000000,0.000000,0.000000,...,0.000000,0.000032,0.008409,0.004459,0.001286,0.000000,0.000000,0.000000,0.002632,0.0000
9,Host 10,0.000000,0.000000,0.0000,0.000000,0,0.00,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000


In [3]:
hostrange_df_to_dict(EOP_hostrange_df)

{'Host 1': {'Grebano_1_host1': 1.0,
  'Ravello_2_host1': 1.0,
  'Etui_3_host1': 1.0,
  'Maxentius_4_host2': 0.0,
  'Licinius_5_host2': 0,
  'Jovian_6_host2': 0.0,
  'Arcadius_7_host2': 0.0,
  'Avitus_8_host3': 0.0,
  'Marcian_9_host3': 0.0,
  'Libius_10_host3': 0.0,
  'Anthemius_11_host3': 0.0,
  'Olybrius_12_host3': 0.0,
  'Phocas_13_host4': 0.0,
  'Caracalla_14_host4': 0.0,
  'Geta_14_host4': 0.0,
  'Leonitus_15_host4': 0,
  'Artabasdos_16_host5': 0.1306532663316583,
  'Rangabe_17_host5': 0.6858974358974359,
  'Staurakios_18_host5': 0.1507692307692308,
  'Bardicus_19_host5': 0.1416309012875537,
  'Quintillus_20_host6': 0,
  'Heraclius_21_host6': 0.0,
  'Heraclonas_22_host6': 0.0,
  'Anivius_23_host6': 0,
  'Komnenos_24_host7': 0,
  'Eudokia_25_host7': 0,
  'Doukas_26_host7': 0.0,
  'Arruntis_27_host7': 0.0,
  'Hostillian_28_host8': 0,
  'Pacatian_nan_host8': 0,
  'Quartinus_29_host9': 0,
  'Bonosus_30_host9': 0,
  'Rozzorie_31_host10': 0,
  'Brede_32_host10': 0,
  'Didius_33_host10':

In [5]:
binarize_host_range(hostrange_df_to_dict(EOP_hostrange_df), continous=False)

{'Host 1': {'Grebano_1_host1': 1,
  'Ravello_2_host1': 1,
  'Etui_3_host1': 1,
  'Maxentius_4_host2': 0,
  'Licinius_5_host2': 0,
  'Jovian_6_host2': 0,
  'Arcadius_7_host2': 0,
  'Avitus_8_host3': 0,
  'Marcian_9_host3': 0,
  'Libius_10_host3': 0,
  'Anthemius_11_host3': 0,
  'Olybrius_12_host3': 0,
  'Phocas_13_host4': 0,
  'Caracalla_14_host4': 0,
  'Geta_14_host4': 0,
  'Leonitus_15_host4': 0,
  'Artabasdos_16_host5': 1,
  'Rangabe_17_host5': 1,
  'Staurakios_18_host5': 1,
  'Bardicus_19_host5': 1,
  'Quintillus_20_host6': 0,
  'Heraclius_21_host6': 0,
  'Heraclonas_22_host6': 0,
  'Anivius_23_host6': 0,
  'Komnenos_24_host7': 0,
  'Eudokia_25_host7': 0,
  'Doukas_26_host7': 0,
  'Arruntis_27_host7': 0,
  'Hostillian_28_host8': 0,
  'Pacatian_nan_host8': 0,
  'Quartinus_29_host9': 0,
  'Bonosus_30_host9': 0,
  'Rozzorie_31_host10': 0,
  'Brede_32_host10': 0,
  'Didius_33_host10': 0,
  'Septimius_34_host10': 0,
  'Diadumenian_35_host10': 0,
  'Elagabalus_36_host10': 0,
  'Pius_37_ho

## Restructure Bact Annotations
Collect necessary information on bacterial annotations, and store them in a uniform way to other data sources

## Restructure Phage Annotations
Collect necessary information on phage annotations, and store them in a uniform way to other data sources